# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available Record Sets and Fields (columns) by their `@id`.

This will help identify what tables and fields are available for extraction and analysis.

In [ ]:
# List record sets and their fields by @id
print("Available record sets (table-like collections):")
for recordset in metadata.record_sets:
    print(f"RecordSet @id: {recordset.id}  | Name: {recordset.name}")
    print("  Fields (columns):")
    for field in recordset.fields:
        print(f"    Field @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")
    print()

# Choose the first record set's @id for demonstration
if metadata.record_sets:
    sample_recordset_id = metadata.record_sets[0].id
    print(f"Sample RecordSet for extraction: {sample_recordset_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing all entities by their `@id`.

We will demonstrate using the first record set.

In [ ]:
# Collect all available record set @id's for extraction
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"All available record_set @id's: {record_set_ids}")

# Extract each record set as a DataFrame, storing by @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display the first record set's columns and a preview
if record_set_ids:
    df = dataframes[record_set_ids[0]]
    print(f"Columns for {record_set_ids[0]}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate numeric field filtering, normalization, and grouping, referencing all fields by their `@id`.

Note: You may wish to cross-reference your record set field list above and update `numeric_field_id` and `group_field_id` accordingly.

In [ ]:
# Example: Choose a record set and a numeric field for demonstration
# (Replace the below IDs with actual values from data overview above if known)

record_set_id = record_set_ids[0]  # Use the first record set
df = dataframes[record_set_id]

# Identify potential numeric fields (float/int columns)
numeric_fields = df.select_dtypes(include=[float, int]).columns.tolist()
print(f"Numeric fields in {record_set_id}: {numeric_fields}")

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field (@id): {numeric_field_id}")

    # Filtering (example: >10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (if any)
    # Identify string/object columns except the numeric field
    groupable_fields = [col for col in df.select_dtypes(include=["object", "category"]).columns if col != numeric_field_id]
    if groupable_fields:
        group_field_id = groupable_fields[0]
        print(f"Grouping by field (@id): {group_field_id}")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Mean of filtered numeric field by group:")
        print(grouped.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA in record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field from the filtered DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`:
- Loaded and inspected available record sets and fields referenced by their `@id`.
- Loaded tabular data from the record set, previewed structure and data types.
- Filtered, normalized, and grouped the data on a numeric field, demonstrating basic EDA.
- Visualized data distributions using matplotlib and seaborn.

You can further extend this notebook by analyzing different fields and record sets (using their `@id`), adapting filters, or building advanced visualizations. For more, see the [mlcroissant documentation](https://mlco.ooo/).